# 4. Backend

This notebook is the **Vitis HLS backend**, end to end: how a scheduled Allo
kernel becomes synthesizable HLS C++, how you pick a target device, configure
top-level interfaces, run C-simulation, drive C-to-RTL synthesis, parse the
synthesis report, and scaffold a project for the hardware flow. Where the CPU
backend only checks *functionality*, the Vitis backend is the path to real
hardware, so it exposes every stage from codegen to bitstream.

This is part 4 of a 4-notebook series: **1. Frontend** (`@kernel`, types,
operators = *what* to compute) -> **2. Scheduling** (`Schedule` primitives =
*how* to map it to hardware) -> **3. Functional simulation** (running kernels on
the CPU backend and Vitis csim) -> **4. Backend** (this notebook: the Vitis HLS
flow). We forward- and back-reference the others where a concept belongs to
them.

In [ ]:

import os
import numpy as np
import tempfile
import allo
from allo.lang import kernel, Stream, range, i32, f32, bf16
from allo.backend.vitis import parse_report
from allo.backend.vitis.utils import is_vitis_available
import os
os.environ["XILINX_VITIS"] = "/tools/Xilinx/Vitis/2023.2" # Set this to your Vitis installation path
print("Vitis toolchain available:", is_vitis_available())

## 1. The Vitis backend and the HLS flow

A scheduled kernel reaches a backend with `s.export(backend, **kwargs)`. The
**CPU** backend (Notebook 3) JITs the kernel through MLIR/LLVM and runs it — it
is *functional only*: it tells you whether the numbers are right, nothing about
timing, area, or clock. The **Vitis** backend takes the same kernel down the
Xilinx HLS flow, one stage at a time:

```
   @kernel + Schedule
        │  s.export("vitis", device=..., freq_mhz=...)
        ▼
  ┌─────────────────────┐
  │ HLS C++ codegen      │  backend.hls_code            no toolchain needed
  └─────────────────────┘  fastest inspection loop; pragmas are visible here
        │
        ▼
  ┌─────────────────────┐
  │ C-simulation (csim)  │  backend(...) / .csim(...)   Python-native shared lib
  └─────────────────────┘  compiles the real HLS C++, checks functional result
        │
        ▼
  ┌─────────────────────┐
  │ C-to-RTL synthesis   │  backend.synth()             Vitis HLS -> csynth.xml
  └─────────────────────┘  first real fmax / latency / resource numbers
        │
        ▼
  ┌─────────────────────┐
  │ hw emulation (hw_emu)│  backend.run("hw_emu")       v++ + XRT, RTL co-sim
  └─────────────────────┘  cycle-accurate; needs a PLATFORM
        │
        ▼
  ┌─────────────────────┐
  │ hardware (hw)        │  backend.run("hw")           v++ bitstream, on board
  └─────────────────────┘  the real thing
```

**When to use each.** Reach for `hls_code` constantly — it is free and shows
exactly how your schedule surfaced as pragmas and loop structure. Use `csim`
to confirm the generated C++ is functionally correct (it exercises the real HLS
kernel, not a Python model). Use `synth` to get the first hardware quality
numbers (fmax, cycles, DSP/LUT/FF/BRAM) without a multi-hour place-and-route.
`hw_emu` and `hw` are the long, platform-locked steps you run from a shell, not
from a tutorial. Everything above `hw_emu` is what we drive in this notebook.

`backend.run(mode, *args)` is the single dispatch point across the whole flow:

In [ ]:
# The mode table that backend.run(mode, ...) dispatches over.
MODES = {
    "csim":   "Python-native C simulation  (== backend(*args) / backend.csim(*args))",
    "csyn":   "C-to-RTL synthesis           (== backend.synth(); takes no run args)",
    "hw_emu": "Hardware emulation via XRT   (needs PLATFORM)",
    "hw":     "Full hardware build + on-board run (needs PLATFORM)",
    "sw_emu": "Deprecated alias; runs csim and warns",
}
for name, desc in MODES.items():
    print(f"{name:8s} -> {desc}")

# Two backends, same kernel: CPU is functional-only, Vitis is the HLS path.
@kernel
def vadd(A: f32[64], B: f32[64], C: f32[64]):
    for i in range(64):
        C[i] = A[i] + B[i]

a = np.arange(64, dtype=np.float32)
b = np.arange(64, dtype=np.float32) * 10
c = np.zeros(64, dtype=np.float32)

vadd.schedule().export("cpu")(a, b, c)            # functional run on the CPU backend
np.testing.assert_allclose(c, a + b)
print("\nCPU backend (functional) OK")

vitis = vadd.schedule().export("vitis")           # the HLS backend object
print("Vitis backend object:", type(vitis).__name__)

## 2. HLS C++ codegen

`s.export("vitis").hls_code` returns the generated synthesizable C++ **as a
string**. It needs no toolchain, so it is the fastest way to see what your
schedule actually produced — every scheduling decision from Notebook 2 surfaces
here as a `#pragma HLS ...` or as loop/array structure. This is the inspection
loop you should live in while tuning a schedule.

Start with the pipelined `vadd`. `s.pipeline(loop, ii=1)` becomes a
`#pragma HLS pipeline II=1` right inside the loop it targets, and the top
function keeps the flat array signature (`float A[64]`).

The snippets below are real slices of that generated C++, shown with a few lines of surrounding context (a `>` marks the line of interest) so you can see where each pragma or type sits — not just the matching line in isolation. Allo also tags every statement with a `// <source>:line:col` provenance comment; those are trimmed here for readability.

In [ ]:
s = vadd.schedule()
s.pipeline(s.loop("i"), ii=1)
code = s.export("vitis").hls_code


def show_context(code, needle, *, before=2, after=3, first_only=True,
                 require=None, strip_comments=True):
    """Print generated-code lines matching `needle`, plus a few lines of context
    above/below so you can see where they sit. `>` marks a match. Allo's per-line
    `// <source>:line:col` provenance comments are trimmed (pass
    strip_comments=False to keep them). `require` keeps only lines that also
    contain it -- e.g. "{" picks a function definition over its forward
    declaration. Uses slice+enumerate, not range(), since
    `range` is Allo's kernel-only range in this notebook."""
    lines = code.splitlines()
    hits = [i for i, ln in enumerate(lines)
            if needle in ln and (require is None or require in ln)]
    if first_only:
        hits = hits[:1]
    shown = set()
    for i in hits:
        lo, hi = max(0, i - before), min(len(lines), i + after + 1)
        for j, ln in enumerate(lines[lo:hi], start=lo):
            if j in shown:
                continue
            shown.add(j)
            if strip_comments:
                pos = ln.find("//")
                if pos > 0:
                    ln = ln[:pos].rstrip()
            print(("> " if needle in lines[j] else "  ") + ln)
    print()


# The pipelined loop in context: the top function, the loop, the pragma the
# schedule injected, and the body it applies to.
show_context(code, "#pragma HLS pipeline", before=4, after=3)
assert "#pragma HLS pipeline II=1" in code

Now a tiled + reordered GEMM. `s.tile(("i","j"), factors=[4,4])` splits the
16-long `i`/`j` loops into 4x4 blocks — that is why the emitted C++ has **five**
`for` loops instead of three (each tiled axis becomes an outer *tile* loop over
a `< 4` inner *point* loop). Pipelining the innermost `k` loop drops the pipeline
pragma exactly there. Reading the C++ back is how you confirm a schedule did what
you meant.

In [ ]:
M = N = K = 16

@kernel
def gemm(A: f32[M, K], B: f32[K, N], C: f32[M, N]):
    for i in range(M):
        for j in range(N):
            for k in range(K):
                C[i, j] += A[i, k] * B[k, j]

gs = gemm.schedule()
gs.tile(("i", "j"), factors=[4, 4])     # 16 = 4 tiles x 4 points, per axis
gs.pipeline(gs.loop("k"), ii=1)
gcode = gs.export("vitis").hls_code

print("void gemm(...) signature present:",
      "void gemm(float A[16][16]" in gcode)
print("number of `for (` loops after tiling:", gcode.count("for ("))
print()

# The full tiled loop skeleton in context: 2 tile loops + 2 point loops wrapping
# the reduction loop, with the II=1 pragma on the innermost.
show_context(gcode, "#pragma HLS pipeline", before=8, after=2)

### Reading the generated code: labels, names, and source locations

Three conventions make the emitted C++ traceable back to your kernel — handy when
reading synthesis reports or debugging codegen.

- **Loop labels — `loop_<iter>_l<line>c<col>`.** Every `for` gets a Vitis HLS
  label built from the loop's **iterator name** and its **source line/column**:
  `loop_i_l6c4:` is the `i` loop written at line 6, column 4. The line/col makes
  sibling loops distinct on their own; a numeric suffix (`loop_i_l5c4_1`) is
  appended *only* when two loops share both name and location — exactly what
  `split` / `tile` produce (the tiled GEMM above emitted `loop_i_l5c4` next to
  `loop_i_l5c4_1`). These labels are what you reference in Vitis loop directives
  and see in per-loop reports.
- **Variable names — source name, else `v<n>`.** A value keeps its **declared
  source name** when it has one (sanitized to a valid C++ identifier and
  uniquified per function; a name colliding with a C++ keyword such as `int` /
  `default` gets a `_1` suffix). Values with no source name — loads and
  intermediate results — become synthetic `v0`, `v1`, …. So below the scalar
  `a` stays `a`, while every load/add is a `v<n>` temporary.
- **Source-location comments — `// <file>:<line>:<col>`.** On by default, each
  statement is tagged with the exact spot in your Python source that produced it
  — the thread from a line of HLS C++ back to the kernel. In a `.py` file it is a
  real line number; in a notebook it points at the cell's temporary file
  (`<string>` / `/tmp/ipykernel_.../…`).

In [ ]:
# The same idea with the // <source>:line:col comments kept (the earlier windows
# trimmed them). Read off: the loop_<iter>_l<line>c<col> label, the v<n>
# temporaries (unnamed loads / arithmetic), and the source thread on every line.
@kernel
def saxpy(a: f32, X: f32[8], Y: f32[8], Z: f32[8]):
    for i in range(8):
        Z[i] = a * X[i] + Y[i]

sx_code = saxpy.schedule().export("vitis").hls_code
show_context(sx_code, "loop_i", before=0, after=6, strip_comments=False)

## 3. Target selection

`export("vitis", ...)` needs a target FPGA. Specify it **either** with a full
part number (`part="xcvu9p-flga2104-2-i"`) **or** with a `device=` shorthand —
not both. The shorthand table:

| `device`           | Part number                     |
| ------------------ | ------------------------------- |
| `ultra96v2`        | `xczu3eg-sbva484-1-i`           |
| `pynqz2`           | `xc7z020clg400-1`               |
| `zedboard`         | `xc7z020clg484-1`               |
| `zcu102`           | `xczu9eg-ffvb1156-2-e`          |
| `zcu104`, `zcu106` | `xczu7ev-ffvc1156-2-e`          |
| `zcu111`           | `xczu28dr-ffvg1517-2MP-e-S`     |
| `vck190`           | `xcvc1902-vsva2197-2MP-e-S`     |
| `vhk158`           | `xcvh1582-vsva3697-2MP-e-S-es1` |
| `u200`             | `xcu200-fsgd2104-2-e`           |
| `u250`             | `xcu250-figd2104-2L-e`          |
| `u280`             | `xcu280-fsvh2892-2L-e`          |
| `u55c`             | `xcu55c-fsvh2892-2L-e`          |

Other constructor knobs: `freq_mhz=300.0` (target clock, drives the timing
constraint synthesis works against), `flow="vitis"` or `"vivado"`, and
`project_path=...` (where on-disk artifacts land). Whichever way you name the
target, the backend resolves it to a concrete `.part`.

In [ ]:
# device= shorthand resolves to the full part on the backend.
be_u280 = gemm.schedule().export("vitis", device="u280")
print("device='u280'  -> backend.part =", be_u280.part)
print("               freq_mhz =", be_u280.freq_mhz, "| flow =", be_u280.flow)

# ...or pass the full part explicitly (same resolved target).
be_part = gemm.schedule().export("vitis", part="xcu280-fsvh2892-2L-e")
print("part='xcu280...' -> backend.part =", be_part.part)
assert be_u280.part == be_part.part

# A different target with a custom clock; the constraint flows into synthesis.
be_fast = gemm.schedule().export("vitis", part="xcvu9p-flga2104-2-i", freq_mhz=250.0)
print("\nVU9P @250MHz -> part =", be_fast.part, "| freq_mhz =", be_fast.freq_mhz)

<img src="icon.png" width=128/>

# Allo: Accelerator Design and Programming Language

## 4. Interface configuration

Before synthesis or a hardware flow, you decide how the top kernel's arguments
connect to the outside world. Three interface kinds cover the common cases,
each addressed by the **zero-based argument index**:

- **`set_axi(index, ...)`** — an `m_axi` master port. This is how a buffer
  argument reads/writes off-chip DRAM/HBM. `offset="slave"` means the base
  address is delivered over the control port; `bundle="gmem"` groups ports onto
  one physical AXI channel (put arguments that stream concurrently on *different*
  bundles to get parallel DRAM bandwidth).
- **`set_axilite(index, ...)`** — an `s_axilite` slave. Used for scalar
  arguments and control registers; pass `-1` for the **return value**.
- **`set_axis(index, ...)`** — an `axis` (AXI-Stream) port for a streaming
  argument.

These map one-to-one onto Vitis HLS `#pragma HLS interface` directives. Here is
a 64-element copy kernel with both buffers on an `m_axi gmem` bundle — grep the
emitted C++ for the pragmas the setters produced:

In [ ]:
@kernel
def axicopy(A: i32[64], B: i32[64]):
    for i in range(64):
        B[i] = A[i] + 1

be = axicopy.schedule().export("vitis", part="xcvu9p-flga2104-2-i")
be.set_axi(0, offset="slave", bundle="gmem")   # arg 0 (A) -> m_axi, bundle gmem
be.set_axi(1, offset="slave", bundle="gmem")   # arg 1 (B) -> m_axi, bundle gmem
acode = be.hls_code

# Both interface pragmas sit directly under the top-function signature:
show_context(acode, "interface mode=m_axi", before=2, after=3)
assert "#pragma HLS interface mode=m_axi port=A offset=slave bundle=gmem" in acode

Scalars and the return value go on `s_axilite`. Below, a kernel that returns a
scalar has its **return value** (`index=-1`) put on the control interface. The
`set_axis(index)` call (for a streaming argument) is analogous; we do not run it
here because a top-level AXI-Stream port needs a streaming argument shape, but
the call form is identical.

In [ ]:
@kernel
def dot(A: i32[64], B: i32[64]) -> i32:
    s: i32 = 0
    for i in range(64):
        s = s + A[i] * B[i]
    return s

bd = dot.schedule().export("vitis", part="xcvu9p-flga2104-2-i")
bd.set_axi(0, bundle="gmem0")       # A on its own DRAM bundle
bd.set_axi(1, bundle="gmem1")       # B on a second bundle (parallel bandwidth)
bd.set_axilite(-1)                  # the scalar return -> s_axilite control port
print("s_axilite pragma emitted:", "mode=s_axilite" in bd.hls_code)
# Illustrative only (needs a Stream argument): bd.set_axis(0, depth=16)

## 5. C-simulation (the backend view)

Notebook 3 introduced csim as a functional check; here is what it *is* on the
backend side. Calling the Vitis backend — `backend(*args)`, or explicitly
`backend.csim(*args)` — lowers the kernel to HLS C++, compiles it into a
**shared library** with the Vitis Clang toolchain, loads it with `ctypes`, and
calls the top function directly from Python. There is no host program: you pass
the same NumPy buffers you use on the CPU backend. Because it compiles the
*real* HLS C++, csim is the reliable functional oracle for the hardware kernel.

One backend detail matters at the boundary: a non-standard `apint` width is
**widened to the next standard C width (8/16/32/64)** at the host ABI, matching
the `generate-apint-wrapper` convention — the synthesizable kernel keeps the true
`ap_int<5>` interface internally, but Python hands it `int8` arrays. We gate the
run on `is_vitis_available()` so the cell passes with or without a toolchain.

In [ ]:
# csim needs the real toolchain: gate + guard so the cell always passes.
if is_vitis_available():
    try:
        s = vadd.schedule()
        s.pipeline(s.loop("i"), ii=1)
        a = np.random.rand(64).astype(np.float32)
        b = np.random.rand(64).astype(np.float32)
        c = np.zeros(64, dtype=np.float32)
        with tempfile.TemporaryDirectory() as proj:
            backend = s.export("vitis", project_path=proj)
            backend(a, b, c)                    # == backend.csim(a, b, c)
        np.testing.assert_allclose(c, a + b, rtol=1e-5)
        print("csim vadd OK  | max abs error vs numpy:", float(np.abs(c - (a + b)).max()))
    except Exception as exc:
        print("csim attempt failed (toolchain issue); skipping:", exc)
else:
    print("Vitis toolchain not available -- skipping csim (codegen cells above still run).")

## 6. Synthesis and report parsing

`backend.synth()` is the heart of the backend: it scaffolds an HLS project,
invokes Vitis HLS to run **C-to-RTL synthesis**, and writes a report. It needs a
target part (from `part=`/`device=`). Two things worth internalizing:

1. **Synthesis and report parsing are decoupled.** `synth()` runs the (slow)
   tool once and leaves `backend.synth_report` pointing at `csynth.xml`. You then
   call `parse_report(backend.synth_report)` as many times as you like — parsing
   is a fast XML read, no re-synth.
2. `parse_report(path)` (the file **or** its directory) returns a `SynthReport`
   whose **fixed fields are typed attributes** (discoverable) while **open-ended
   collections are dicts/lists** (`r.interfaces`, `r.modules`). BRAM counts are in
   18K-block units. `print(r)` gives a readable one-screen summary.

Below we run **one small real synthesis** — a 16x16x16 GEMM, tiled and pipelined
— inside a temporary directory. It finishes in well under a minute on this part.
The whole thing is gated on `is_vitis_available()` and wrapped in `try/except`:
on success we parse the report and store it for the capstone; if Vitis is missing
or synthesis fails, we print a note and the cell still passes.

In [ ]:
PART = "xcvu9p-flga2104-2-i"   # a mid-size UltraScale+ part; small kernel synths fast

SYNTH_REPORT = None            # capstone (section 9) reuses this -- no second synth

@kernel
def gemm_syn(A: f32[16, 16], B: f32[16, 16], C: f32[16, 16]):
    for i in range(16):
        for j in range(16):
            for k in range(16):
                C[i, j] += A[i, k] * B[k, j]

if is_vitis_available():
    try:
        s = gemm_syn.schedule()
        s.tile(("i", "j"), factors=[4, 4])
        s.pipeline(s.loop("k"), ii=1)
        with tempfile.TemporaryDirectory() as proj:
            mod = s.export("vitis", part=PART, project_path=proj)
            mod.synth()                              # <-- the one real synthesis
            print("synth report at:", mod.synth_report, "| exists:", mod.synth_report.exists())
            # Parse INSIDE the with-block: parse_report returns an in-memory
            # object that outlives the temp dir.
            SYNTH_REPORT = parse_report(mod.synth_report)
        print("\nSynthesis + parse succeeded.\n")
    except Exception as exc:
        print("Synthesis skipped/failed (needs a full Vitis install):", exc)
        print("The flow above is exactly how you would drive it.")
else:
    print("Vitis toolchain not available -- synthesis needs a full Vitis install;")
    print("the code above is the flow. parse_report() usage is shown below on the object.")

`print(r)` first — the human-readable digest — then reach into the typed fields.
Note how `r.timing`, `r.latency`, `r.resources`, `r.available`, and
`r.utilization` group related numbers, while `r.interfaces` and `r.modules` are
iterable collections (the per-module breakdown includes the top module itself).

In [ ]:
r = SYNTH_REPORT
if r is None:
    print("No report available (synthesis was skipped). Field access below is illustrative:")
    print("  r.fmax, r.latency.worst_cycles, r.resources.dsp/.lut/.ff/.bram, r.utilization,")
    print("  and iterate r.interfaces (name/protocol/data_bits) and r.modules[name].")
else:
    print("========== print(r) ==========")
    print(r)
    print("========== typed fields ==========")
    print("version / part / family / top :", r.version, "/", r.part, "/",
          r.product_family, "/", r.top)
    print("fmax (MHz)                    :", round(r.fmax, 2))
    print("clock target / estimated (ns) :", r.timing.target_clock_ns, "/",
          r.timing.estimated_clock_ns)
    print("latency worst cycles / time   :", r.latency.worst_cycles, "/",
          r.latency.worst_time)
    print("latency interval_min / type   :", r.latency.interval_min, "/",
          r.latency.pipeline_type)
    print("resources dsp/lut/ff/bram/uram:", r.resources.dsp, r.resources.lut,
          r.resources.ff, r.resources.bram, r.resources.uram)
    print("device capacity (lut)         :", r.available.lut)
    print("utilization (%)               :",
          {k: round(v, 3) for k, v in r.utilization.items()})

    print("\n-- HW interfaces (grouped per bundle) --")
    for itf in r.interfaces:
        print(f"   {itf.name:10s} {itf.protocol:14s} data_bits={itf.data_bits}")

    print("\n-- modules (per-module breakdown, incl. top) --")
    for name, m in r.modules.items():
        print(f"   {name:36s} dsp={m.resources.dsp:>3}  "
              f"II={m.latency.pipeline_ii}  fmax={round(m.timing.fmax_mhz, 1)}MHz")

## 7. Project scaffolding and the hardware flows

csim and synthesis are self-contained. The **emulation and hardware** flows
(`hw_emu`, `hw`) go through `v++` and an XRT host, so they need a platform
exported in the environment (`export PLATFORM=/path/to/<shell>.xpfm`) and can take
hours to link. You do not drive those from a notebook — you scaffold a project and
run `make` in a shell. The relevant API:

- **`backend.scaffold_project(path, *args)`** materializes a full project at
  `path`: the kernel, a generated Makefile, an XRT host, and your `*args` packed
  into binary input files for emulation/hardware. Then, in a shell:

  ```bash
  cd path
  make csim      # or: make csyn / make hw_emu / make hw   (see ENVIRONMENT.md)
  ```

- **`backend.precheck(mode)`** scaffolds the kernel `.xo`, the XRT host, and (for
  emulation) the emconfig, and validates that the project *builds* — **without**
  the multi-hour, platform-locked link step. Use it to confirm the frontend
  produced a buildable project before committing to a full `hw_emu`/`hw` run.

- **`backend.set_csim_override(**vars)`** overrides C-simulation Makefile
  variables (e.g. `cxx`, `hls_cxxflags`) when you need a non-default compiler or
  flags.

The cell below is **illustrative** — it only prints the commands you would run;
it does not invoke `v++` or build hardware (those are out of scope for a
tutorial and require a platform).

In [ ]:
# ILLUSTRATIVE: we print the flow rather than run v++/XRT.
flow = """
# 1. In Python: pick a target, configure interfaces, scaffold with sample inputs.
A = np.arange(64, dtype=np.float32)
B = np.arange(64, dtype=np.float32) * 10
C = np.zeros(64, dtype=np.float32)

be = vadd.schedule().export("vitis", device="u280", project_path="/path/to/proj")
be.set_axi(0, bundle="gmem0")
be.set_axi(1, bundle="gmem1")
be.set_axi(2, bundle="gmem2")
be.scaffold_project("/path/to/proj", A, B, C)   # packs A, B, C into binary files

# 2. Validate a buildable project WITHOUT the long link step:
be.precheck("hw_emu")

# 3. In a shell:
#      export PLATFORM=/path/to/<shell>.xpfm
#      cd /path/to/proj
#      make run TARGET=hw_emu         # cycle-accurate RTL co-simulation via XRT
#      make xclbin TARGET=hw          # full bitstream build + on-board run
"""
print(flow)
print("(Not executed here -- hw_emu/hw need a platform and take hours to link.)")

# set_csim_override lets you swap the csim compiler / flags when needed:
print("set_csim_override example: be.set_csim_override(cxx='g++', hls_cxxflags='-O2')")

## 8. Environment variables

A few environment variables tune the backend:

- **`XILINX_VITIS`** — path to the Vitis install. Setting it lets Allo find the
  toolchain without you sourcing the full Vitis setup scripts.
- **`VIVADO_IMPL_JOBS`** (default `4`) — parallelism for the Vivado
  implementation step during hardware builds.
- **`ALLO_ENABLE_VITIS_APFLOAT`** (default `0`) — force-enable `ap_float`
  codegen. `ap_float` is required for **`bf16`/`tf32`** and needs Vitis 2023.1+ to
  build. When unset, Allo auto-detects the Vitis version and enables it when
  supported.

Below is a **codegen-only** demonstration (no toolchain): with apfloat enabled, a
`bf16` kernel emits `ap_float<16,8>` (16-bit float, 8-bit exponent) for its
buffers. We set the variable, generate, check, then restore it.

In [ ]:
prev = os.environ.get("ALLO_ENABLE_VITIS_APFLOAT")
os.environ["ALLO_ENABLE_VITIS_APFLOAT"] = "1"      # force ap_float codegen
try:
    @kernel
    def add_bf16(A: bf16[16], B: bf16[16], C: bf16[16]):
        for i in range(16):
            C[i] = A[i] + B[i]

    bf16_code = add_bf16.schedule().export("vitis").hls_code
    print("ap_float<16,8> in the bf16 signature:", "ap_float<16,8>" in bf16_code)
    print()
    # bf16 lowers to ap_float<16,8> -- visible in the signature and the temporaries:
    show_context(bf16_code, "ap_float<16,8>", before=0, after=5, require="{")
finally:
    # Restore the environment exactly as we found it.
    if prev is None:
        os.environ.pop("ALLO_ENABLE_VITIS_APFLOAT", None)
    else:
        os.environ["ALLO_ENABLE_VITIS_APFLOAT"] = prev

## 9. Worked example: GEMM end-to-end, and per-PE specialization

Pulling the flow together on the 16x16x16 GEMM: we **schedule** it (tile +
pipeline, from Notebook 2), **inspect** the HLS pragmas (section 2), **csim** it
for functional correctness (section 5), and read **fmax / latency / resources**
off the synthesis report we already produced in section 6 (no second synth). This
is the loop you run for real designs: change the schedule, re-inspect codegen,
re-csim, re-synth, compare the numbers.

In [ ]:
# A fresh kernel: export() mutates the kernel's module, so section 6's gemm_syn
# is already tiled -- we schedule a clean copy here.
@kernel
def gemm_cap(A: f32[16, 16], B: f32[16, 16], C: f32[16, 16]):
    for i in range(16):
        for j in range(16):
            for k in range(16):
                C[i, j] += A[i, k] * B[k, j]

cap = gemm_cap.schedule()
cap.tile(("i", "j"), factors=[4, 4])       # schedule once...
cap.pipeline(cap.loop("k"), ii=1)

# (a) inspect codegen  +  (b) csim -- from a single backend (one export).
if is_vitis_available():
    try:
        A = np.random.rand(16, 16).astype(np.float32)
        B = np.random.rand(16, 16).astype(np.float32)
        C = np.zeros((16, 16), dtype=np.float32)
        with tempfile.TemporaryDirectory() as proj:
            be = cap.export("vitis", part=PART, project_path=proj)
            cap_code = be.hls_code
            print("[codegen] tiled loop count :", cap_code.count("for ("),
                  "| pipeline pragma:", "#pragma HLS pipeline II=1" in cap_code)
            be(A, B, C)                     # csim on the same backend
        np.testing.assert_allclose(C, A @ B, rtol=1e-3, atol=1e-3)
        print("[csim]    GEMM matches numpy  | max err:", float(np.abs(C - A @ B).max()))
    except Exception as exc:
        print("[csim]    skipped:", exc)
else:
    cap_code = cap.export("vitis", part=PART).hls_code
    print("[codegen] tiled loop count :", cap_code.count("for ("),
          "| pipeline pragma:", "#pragma HLS pipeline II=1" in cap_code)
    print("[csim]    Vitis not available -- skipped")

# (c) hardware numbers -- reuse the section-6 report object (no re-synth)
if SYNTH_REPORT is not None:
    r = SYNTH_REPORT
    print(f"[synth]   fmax={round(r.fmax,1)}MHz  "
          f"latency={r.latency.worst_cycles} cyc ({r.latency.worst_time})  "
          f"DSP={r.resources.dsp} LUT={r.resources.lut} "
          f"FF={r.resources.ff} BRAM={r.resources.bram}")
else:
    print("[synth]   no report (synthesis was skipped in section 6)")

### Per-PE specialization for spatial architectures

The GEMM above is a single pipelined datapath. A **spatial** design instead lays
out an explicit array of Processing Elements (PEs) — a systolic array. In Allo you
write one PE body with a `mapping=[...]` and use `allo.get_wid(dim)` to read the
PE's coordinates (Notebook 2's spatial mapping). The backend **specializes each
PE into its own C++ function** by constant-folding those coordinates: interior PEs
that do the MAC work keep their full body, while idle corner PEs are **pruned
entirely**.

We build the 2x2 output-stationary array from the test suite (a 4x4 PE grid: a
2x2 compute core ringed by feeder/drain PEs) and inspect the emitted C++. The
active interior PE `systolic_2d_pe_1_1` has its own function; the idle corner
`systolic_2d_pe_0_0` does not appear at all. That per-coordinate pruning is what
makes a spatial dataflow architecture efficient — each PE compiles to exactly the
logic its position needs, nothing more.

In [ ]:
M2, N2, K2 = 2, 2, 2
P0, P1 = M2 + 2, N2 + 2   # a 4x4 PE grid ringing a 2x2 compute core

@kernel
def systolic_2d(A: f32[M2, K2], B: f32[K2, N2], C: f32[M2, N2]):
    fifo_A: Stream[f32][P0, P1]
    fifo_B: Stream[f32][P0, P1]

    @kernel(mapping=[P0, P1])
    def pe(
        A: f32[M2, K2], B: f32[K2, N2], C: f32[M2, N2],
        fifo_A: Stream[f32][P0, P1], fifo_B: Stream[f32][P0, P1],
    ):
        i = allo.get_wid(0)
        j = allo.get_wid(1)
        if (i == 0 or i == M2 + 1) and (j == 0 or j == N2 + 1):
            pass                                   # idle corner PE
        elif j == 0:
            for k in range(K2):
                fifo_A[i, j + 1].put(A[i - 1, k])  # feed A rows
        elif i == 0:
            for k in range(K2):
                fifo_B[i + 1, j].put(B[k, j - 1])  # feed B cols
        elif i == M2 + 1:
            for k in range(K2):
                b: f32 = fifo_B[i, j].get()        # drain B
        elif j == N2 + 1:
            for k in range(K2):
                a: f32 = fifo_A[i, j].get()        # drain A
        else:
            c: f32 = 0                             # interior compute PE
            for k in range(K2):
                a: f32 = fifo_A[i, j].get()
                b: f32 = fifo_B[i, j].get()
                c += a * b
                fifo_A[i, j + 1].put(a)
                fifo_B[i + 1, j].put(b)
            C[i - 1, j - 1] = c

    pe(A, B, C, fifo_A, fifo_B)

sys_code = systolic_2d.schedule().export("vitis").hls_code
print("interior compute PE  systolic_2d_pe_1_1 emitted :",
      "void systolic_2d_pe_1_1(" in sys_code)
print("idle corner PE       systolic_2d_pe_0_0 pruned  :",
      "systolic_2d_pe_0_0" not in sys_code)
print("edge feeder PE       systolic_2d_pe_0_1 emitted :",
      "systolic_2d_pe_0_1(" in sys_code)

import re
pes = sorted(set(re.findall(r"systolic_2d_pe_\d_\d", sys_code)))
print("\nspecialized PE functions actually emitted:")
print("  ", pes)

# What one specialized PE actually looks like -- the interior compute PE, in context:
show_context(sys_code, "void systolic_2d_pe_1_1(", before=0, after=16, require="{")

### Wrap-up

That is the Vitis backend end to end: `export("vitis")` gives you a backend object;
`hls_code` shows the synthesizable C++ (and your schedule's pragmas) with no
toolchain; `device=`/`part=` pick the target; `set_axi`/`set_axilite`/`set_axis`
wire up the interfaces; calling the backend runs csim against real HLS C++;
`synth()` produces `csynth.xml` and `parse_report` turns it into typed
fmax/latency/resource numbers; and `scaffold_project` + `make` (with a `PLATFORM`)
carry you into emulation and hardware. Combined with the frontend (NB1),
scheduling (NB2), and functional simulation (NB3), you now have the whole path
from a Python `@kernel` to numbers on real silicon.